# DeepSeek V4.1 Flash (Abliterated) — Google Colab Web Runner
Runs `leonxger/deepseek-colab` with hardware-adaptive **G4 (96GB)** / **A100 High-RAM (80GB / 40GB)** layouts and exposes a private OpenAI-compatible API via `cloudflared`.
1. Open `Runtime > Change runtime type` -> select **G4** (best, 105-118 tok/s) or **A100 + High-RAM**.
2. Run Cell 1 (~4-6 min cold boot) to get your `https://....trycloudflare.com/v1` Base URL + API key.


In [ ]:
#@title 1. Boot DeepSeek V4.1 Flash (Abliterated) + Cloudflare Tunnel
#@markdown **Runtime:** Select **G4** or **A100 (High-RAM)** via `Runtime > Change runtime type`.
API_KEY = "" #@param {type:"string"}
HF_TOKEN = "" #@param {type:"string"}
UNCENSOR = True #@param {type:"boolean"}
IMAGES = True #@param {type:"boolean"}
CONTEXT_TOKENS = 262144 #@param [32768, 65536, 131072, 262144] {type:"raw"}
FAST_MODE = False #@param {type:"boolean"}

import os, re, secrets, shutil, subprocess, time, pathlib

def sh(cmd, check=True):
    print(f"$ {cmd}", flush=True)
    r = subprocess.run(cmd, shell=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
    if r.stdout:
        print(r.stdout.rstrip(), flush=True)
    if check and r.returncode != 0:
        raise RuntimeError(f"Command failed ({r.returncode}): {cmd}")
    return r.returncode

print("=== Hardware ===")
sh("nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv,noheader && free -h && df -h /content && nproc", check=False)

if not pathlib.Path("/content/deepseek-colab/.git").exists():
    sh("git clone --depth 1 https://github.com/leonxger/deepseek-colab.git /content/deepseek-colab")

sh("rm -rf /content/vm && cp -a /content/deepseek-colab/vm /content/vm")

BOOT_SH = '#!/bin/bash\n# Cold start on a Colab G4: weights into RAM, llama.cpp built, server up, VRAM copies freed, Engram filled, then the gate opens port 18080.\nset -uo pipefail\n: "${CTX:?} ${UBATCH:?} ${MOE_CACHE_MIB:?} ${DYN:?} ${MMPROJ:?}"\nexport RUNG=IQ2_XXS PORT=18081 DRAFT=${DRAFT:-1} UNCENSOR=${UNCENSOR:-1} WORKERS=${WORKERS:-64}\nexport LLAMA_DSV41_CED=1 LLAMA_DSV41_CED_SPEC=1\nexport PIN=1 PIN_MAPPED=1 CACHE_RAM=1024 LLAMA_MOE_PAD_ZERO=1 LLAMA_MOE_CACHE_PIN_PREFILL=1 LLAMA_MOE_CACHE_DYN=$DYN\nexport LLAMA_MOE_CACHE_PROFILE=/content/vm/profile-$RUNG.txt LLAMA_MOE_CACHE_RESIDENT=/content/res/resident.txt\nexport CPU_RE=$(python3 -c "print(r\'blk\\.(%s)\\.ffn_(up|gate|down)_exps\\.weight=CPU\' % \'|\'.join(map(str, range(40))))")\n[ -s /content/secrets/hf_hdr ] && export HF_HDR=/content/secrets/hf_hdr\nHF=https://huggingface.co/ISTA-DASLab/DeepSeek-V4.1-Flash-1017-GSQ-RCO-GGUF/resolve/a419d74cd28d5b9f8c5e2b9077d11a609c17bc32\nNAME=DeepSeek-V4.1-Flash-1017-GSQ-RCO-$RUNG\nENGRAM=DeepSeek-V4.1-Flash-Engram-IQ4_NL.gguf\nDRAFT_SHA=6ba6ac1b331c540691a209e7fc97dec2a24e295acb1e755c84c7d24a4373927a\nM=/mnt/ram/m; R=/content/res; V=/content/vm\nmkdir -p $R /mnt/ram /content/model /content/llama\nrm -f $R/resident.txt $R/READY\nT0=$(date +%s)\nst() { echo "$(( $(date +%s) - T0 ))s $(date +%T) $*" >> $R/timeline.txt; }\nseen() { grep -qE "$1" $R/timeline.txt; }\nhdr() { echo "Authorization: Bearer $(cat /content/secrets/key)"; }\nst "boot start ctx=$CTX ubatch=$UBATCH cache=$MOE_CACHE_MIB dyn=$DYN images=$MMPROJ draft=$DRAFT uncensored=$UNCENSOR"\nVRAM_MIB=$(nvidia-smi --query-gpu=memory.total --format=csv,noheader,nounits 2>/dev/null | head -1 | tr -dc \'0-9\')\nRAM_GB=$(awk \'/MemTotal/ {print int($2/1048576)}\' /proc/meminfo)\nDISK_FREE_GB=$(df -BG --output=avail /content | tail -1 | tr -dc \'0-9\')\nVRAM_MIB=${VRAM_MIB:-0}; RAM_GB=${RAM_GB:-0}; DISK_FREE_GB=${DISK_FREE_GB:-0}\nst "hw vram=${VRAM_MIB}MiB ram=${RAM_GB}GB disk=${DISK_FREE_GB}GB"\nENGRAM_DIR=$M; VRAM_GUARD=60000; BACKBONE_ON_DISK=0; SKIP_ENGRAM=0; PIN_COLD=1\nif [ "$VRAM_MIB" -ge 90000 ] && [ "$RAM_GB" -ge 180 ]; then\n  st "layout G4 (RAM Engram)"\nelif [ "$VRAM_MIB" -ge 75000 ] && [ "$RAM_GB" -ge 155 ]; then\n  export MOE_CACHE_MIB=52000 DYN=160 DMA=0\n  [ "$CTX" -gt 262144 ] && export CTX=262144\n  ENGRAM_DIR=/content/engram; VRAM_GUARD=45000\n  [ "$DISK_FREE_GB" -lt 115 ] && { st "GUARD: A100-80G needs 115 GB free disk for Engram (have ${DISK_FREE_GB} GB) FAILED"; exit 1; }\n  st "layout A100-80G (disk Engram mmap, cache=${MOE_CACHE_MIB}MiB, dyn=${DYN}, ctx=${CTX})"\nelif [ "$VRAM_MIB" -ge 38000 ] && [ "$RAM_GB" -ge 78 ]; then\n  export MOE_CACHE_MIB=17000 DYN=48 DMA=0 DRAFT=0 MMPROJ=0\n  [ "$CTX" -gt 131072 ] && export CTX=131072\n  ENGRAM_DIR=/content/engram; VRAM_GUARD=20000; BACKBONE_ON_DISK=1; SKIP_ENGRAM=1; PIN_COLD=0\n  [ "$DISK_FREE_GB" -lt 142 ] && { st "GUARD: A100-40G backbone needs 142 GB free disk (have ${DISK_FREE_GB} GB) FAILED"; exit 1; }\n  st "layout A100-40G (disk backbone mmap, sparse Engram, cache=${MOE_CACHE_MIB}MiB, ctx=${CTX})"\nelse\n  st "GUARD: unsupported GPU/RAM (${VRAM_MIB}MiB VRAM, ${RAM_GB}GB RAM) FAILED"; exit 1\nfi\nmountpoint -q /mnt/ram || mount -t tmpfs -o size=$(( $(awk \'/MemTotal/ {print $2}\' /proc/meminfo) / 1000000 - 6 ))G tmpfs /mnt/ram\nmkdir -p $M "$ENGRAM_DIR" /content/model_disk\n\nfetch() {  # $1 = backbone|engram|draft\n  local dst=$M\n  [ "$1" = "engram" ] && dst=$ENGRAM_DIR\n  [ "$1" = "backbone" ] && [ "$BACKBONE_ON_DISK" = 1 ] && dst=/content/model_disk\n  python3 $V/jobs.py $1 $dst $R/j_$1.json >> $R/fetch.log 2>&1 || return 1\n  python3 $V/pfetch.py $R/j_$1.json --workers $WORKERS --chunk-mb 64 --tag $1 --log $R/fetch.log\n}\n\n{ bash $V/build.sh > $R/llama.log 2>&1 && st "llama.cpp built" || st "llama.cpp build FAILED"; } &\n[ "$UNCENSOR" = 1 ] && { python3 -I $V/mkpatch.py /content/patch > $R/patch.log 2>&1 && st "uncensor patch fetched" || st "uncensor patch FAILED"; } &\n[ "$MMPROJ" = 1 ] && { MMPROJ_FETCH_ONLY=1 timeout 300 [ -f $V/libpin.so ] || gcc -shared -fPIC -O2 $V/pin.c -o $V/libpin.so -ldl -lpthread 2>/dev/null || true\nbash $V/serve.sh > $R/mmproj.log 2>&1 && st "image encoder fetched" || st "image encoder FAILED"; } &\n{ fetch backbone && st "backbone fetched" || st "backbone FAILED"; } &\n[ "$DRAFT" = 1 ] && { fetch draft && sha256sum $M/dspark.gguf | grep -q "^$DRAFT_SHA" && st "draft fetched" || st "draft FAILED"; } &\nwait\nseen FAILED && { st "fetch FAILED, stopping"; exit 1; }\n( for k in $(seq 1 900); do [ -f $R/READY ] && exit 0; a=$(awk \'/MemAvailable/{print int($2/1024)}\' /proc/meminfo)\n    [ "$a" -lt 1200 ] && { for p in $(pgrep -f "[p]fetch.py"); do kill $p; done; st "GUARD: ${a} MiB RAM available, fetch stopped FAILED"; exit 0; }; sleep 2; done ) &\nif [ "$BACKBONE_ON_DISK" = 1 ]; then\n  for i in 1 2 3 4 5; do ln -sf /content/model_disk/$NAME-0000$i-of-00006.gguf /content/model/$NAME-0000$i-of-00006.gguf; done\nelse\n  for i in 1 2 3 4 5; do ln -sf $M/$NAME-0000$i-of-00006.gguf /content/model/$NAME-0000$i-of-00006.gguf; done\nfi\nln -sf "$ENGRAM_DIR/$ENGRAM" /content/model/$NAME-00006-of-00006.gguf\npython3 -I -c "import os; f=os.open(\'$ENGRAM_DIR/$ENGRAM\', os.O_WRONLY|os.O_CREAT, 0o644); os.ftruncate(f, 110595290816); os.close(f)"\nfor k in 1 2 3 4 5; do  # the sparse Engram needs its real header before the server starts; the table itself is filled after the punch\n  curl -sfL --retry 3 ${HF_HDR:+-H @$HF_HDR} -r 0-1048575 -o $R/engram_head.bin "$HF/$ENGRAM" && [ "$(head -c 4 $R/engram_head.bin)" = GGUF ] && break\n  sleep 3\ndone\n[ "$(head -c 4 $R/engram_head.bin 2>/dev/null)" = GGUF ] || { st "engram header FAILED"; exit 1; }\ndd if=$R/engram_head.bin of="$ENGRAM_DIR/$ENGRAM" conv=notrunc status=none\npython3 -I $V/checksizes.py /content/model > $R/sizes.txt 2>&1 && st "sizes ok" || { st "sizes FAILED"; exit 1; }\nif [ "$UNCENSOR" = 1 ]; then\n  python3 -I $V/applypatch.py /content/model /content/patch >> $R/patch.log 2>&1 && st "uncensored tensors applied" || { st "patch apply FAILED"; exit 1; }\nfi\n[ -f $V/libpin.so ] || gcc -shared -fPIC -O2 $V/pin.c -o $V/libpin.so -ldl -lpthread 2>/dev/null || true\nbash $V/serve.sh\nst "server started"\nfor k in $(seq 1 900); do\n  curl -sf -o /dev/null -H "$(hdr)" http://127.0.0.1:$PORT/health && { st "healthy"; break; }\n  [ -d /proc/$(cat $R/server.pid) ] || { st "server died"; exit 1; }\n  sleep 1\ndone\nVRAM=$(nvidia-smi --query-gpu=memory.used --format=csv,noheader,nounits | head -1)\ngrep -E "model buffer size|KV buffer size|compute buffer size|MoE cache" $R/server.log > $R/buffers.txt\n# punching a tensor the server still reads from RAM would zero live weights, so check that the weights went to the GPU first\nif [ "$VRAM" -lt "$VRAM_GUARD" ] || ! grep -q "CUDA0 model buffer size" $R/buffers.txt; then st "GUARD: VRAM ${VRAM} MiB (< ${VRAM_GUARD}), no CUDA model buffer FAILED"; exit 1; fi\n[ -s $R/resident.txt ] && grep -q "prefilled" $R/server.log || { st "GUARD: no resident expert list FAILED"; exit 1; }\nif [ "$BACKBONE_ON_DISK" = 1 ]; then\n  st "punch/pin: skipped (disk mmap mode, VRAM ${VRAM} MiB)"\nelse\n  PUNCH_FILES="$M/$NAME-0000[2-5]-of-00006.gguf"\n  python3 -I $V/punch.py "$CPU_RE" $PUNCH_FILES > $R/punch.txt 2>&1 && st "punched $(cat $R/punch.txt), VRAM ${VRAM} MiB" || { st "punch FAILED"; exit 1; }\n  python3 $V/punch.py --experts $R/resident.txt $PUNCH_FILES > $R/punch_res.txt 2>&1 && st "$(cat $R/punch_res.txt)" || { st "resident punch FAILED"; exit 1; }\n  if [ "$DRAFT" = 1 ] && [ "$(grep -c \'CUDA0 model buffer size\' $R/buffers.txt)" -ge 2 ]; then\n    python3 -I $V/punch.py \'NO_CPU_TENSOR=CPU\' $M/dspark.gguf > $R/punch_draft.txt 2>&1 && st "draft punched $(cat $R/punch_draft.txt)"\n  fi\n  if [ "$PIN_COLD" = 1 ] && [ -f "$V/libpin.so" ]; then\n    python3 $V/punch.py --pin-cold $R/pin_request.tmp $R/resident.txt $PUNCH_FILES > $R/pin.txt 2>&1 && mv $R/pin_request.tmp $R/pin_request\n    for k in $(seq 1 600); do grep -q DONE $R/pin_result 2>/dev/null && break; sleep 1; done\n    st "pin: $(tail -1 $R/pin_result 2>/dev/null)"\n  fi\nfi\nif [ "$SKIP_ENGRAM" = 1 ]; then\n  st "engram skipped (sparse mode for A100-40G)"\nelif [ "$(du -BG "$ENGRAM_DIR/$ENGRAM" 2>/dev/null | awk \'{print int($1)}\')" -ge 100 ]; then\n  st "engram cached on disk"\nelse\n  fetch engram && st "engram done" || { st "engram FAILED"; exit 1; }\nfi\nfree -b > $R/free_after_engram.txt\ntouch $R/READY\ncurl -sf -H "$(hdr)" http://127.0.0.1:$PORT/v1/chat/completions -H "Content-Type: application/json" \\\n  -d \'{"messages":[{"role":"user","content":"Say OK."}],"max_tokens":8,"reasoning_effort":"low"}\' > $R/warmup.json && st "first token (warmup done)" || { st "warmup FAILED"; exit 1; }\nsetsid nohup python3 $V/gate.py 18080 $PORT > $R/gate.log 2>&1 < /dev/null &\nfor k in $(seq 1 50); do grep -q "gate open" $R/gate.log 2>/dev/null && break; sleep 0.1; done\ngrep -q "gate open" $R/gate.log && st "gate open 18080" || st "gate FAILED"\n'
BUILD_SH = '#!/bin/bash\n# Builds llama.cpp b11963 for the detected GPU arch with the patches in patches/series into /content/llama/bin, about 2 minutes.\nset -uo pipefail\nCOMMIT=39febd2adb947b70da35e95f9351b3bf58ad170a\nR=/content/res; P=/content/vm/patches; S=/content/src/llama.cpp; D=/content/llama/bin\nst() { echo "$(date +%T) $*" >> $R/build.txt; }\nif [ -x "$D/llama-server" ]; then st "build cached"; exit 0; fi\nst "build start"\ncommand -v cmake > /dev/null || pip install -q cmake > $R/build_cmake.log 2>&1\nmkdir -p /content/src\nif [ -d "$S/.git" ]; then\n  git -C "$S" checkout -- . && git -C "$S" clean -fd > /dev/null 2>&1\nelse\n  git clone -q --depth 1 --branch b11963 https://github.com/ggml-org/llama.cpp.git $S || { st "clone FAILED"; exit 1; }\nfi\n[ "$(git -C $S rev-parse HEAD)" = "$COMMIT" ] || { st "commit mismatch $(git -C $S rev-parse HEAD) FAILED"; exit 1; }\nfor p in $(grep -v \'^#\' $P/series); do python3 $P/$p $S >> $R/build.txt 2>&1 || { st "patch $p FAILED"; exit 1; }; done\nexport PATH=/usr/local/cuda/bin:$PATH\nCUDA_ARCH=$(nvidia-smi --query-gpu=compute_cap --format=csv,noheader 2>/dev/null | head -1 | tr -d \'. \')\n[ -n "$CUDA_ARCH" ] || CUDA_ARCH=120\nJOBS=$(nproc 2>/dev/null || echo 16)\nst "cmake arch=$CUDA_ARCH jobs=$JOBS"\nsed -i \'s/FATAL_ERROR/STATUS/g\' "$S/scripts/ui-assets.cmake" 2>/dev/null || true\ncmake -S $S -B $S/build -DGGML_CUDA=ON -DCMAKE_CUDA_ARCHITECTURES=$CUDA_ARCH -DLLAMA_BUILD_TESTS=ON -DLLAMA_CURL=OFF -DLLAMA_OPENSSL=ON \\\n  -DLLAMA_BUILD_UI=OFF \\\n  -DCMAKE_BUILD_TYPE=Release > $R/build_cmake.log 2>&1 || { st "cmake FAILED: $(tail -3 $R/build_cmake.log | tr \'\\n\' \' \')"; exit 1; }\nnice -n 5 cmake --build $S/build -j "$JOBS" --target llama-server > $R/build_make.log 2>&1 || { st "make FAILED: $(grep -m3 -i error $R/build_make.log | tr \'\\n\' \' \')"; exit 1; }\nmkdir -p $D && cp -a $S/build/bin/* $D/ && find $S/build -name "*.so*" -exec cp -a {} $D/ \\;\nfor l in libcudart.so libcublas.so libcublasLt.so; do cp -a /usr/local/cuda/lib64/$l* $D/ 2>/dev/null; done\nst "build done"\n'
SERVE_SH = '#!/bin/bash\n# Starts llama-server detached with the tuned flags; MMPROJ_FETCH_ONLY=1 only downloads the image encoder.\nMMP=/content/mmproj/mmproj-DeepSeek-V4.1-Flash-1017-BF16.gguf\nif [ "${MMPROJ:-0}" = 1 ]; then  # image encoder on local disk (the RAM disk has no room), pinned HF revision, size and sha256 checked\n  MM_SIZE=972208096; MM_SHA=0fdc4b7693f287e9665940cf42bd39c546ffe4f0def89c569d7d0e45d60d2df1\n  mm_ok() { [ "$(stat -c %s $MMP 2>/dev/null)" = $MM_SIZE ] && sha256sum $MMP | grep -q "^$MM_SHA"; }\n  mkdir -p /content/mmproj /content/res\n  if ! mm_ok; then\n    rm -f $MMP\n    echo "[{\\"url\\": \\"https://huggingface.co/ISTA-DASLab/DeepSeek-V4.1-Flash-1017-GSQ-RCO-GGUF/resolve/a419d74cd28d5b9f8c5e2b9077d11a609c17bc32/$(basename $MMP)\\", \\"dest\\": \\"$MMP\\", \\"dest_size\\": $MM_SIZE, \\"src_off\\": 0, \\"dest_off\\": 0, \\"size\\": $MM_SIZE}]" > /content/res/j_mmproj.json\n    python3 /content/vm/pfetch.py /content/res/j_mmproj.json --workers 16 --chunk-mb 64 --tag mmproj && mm_ok || { echo "image encoder fetch or check FAILED"; exit 1; }\n  fi\n  [ "${MMPROJ_FETCH_ONLY:-0}" = 1 ] && exit 0\nfi\nBIN=/content/llama/bin/llama-server\nexport LLAMA_MOE_DEVCACHE=1 GGML_CUDA_HCFUSE=1 GGML_SCHED_ASYNC_INPUTS=1 GGML_CUDA_GRAPH_FIRST_CHUNK=32 LLAMA_KV_SEQRM_INDEX=1 GGML_CUDA_MOE_COPY_ALIGN=1\nif [ "$MMPROJ" = 1 ] || [ "${DMA:-0}" != 1 ]; then  # misses by the copy engine stall on image prompts and were measured only in the fast layout\n  export LLAMA_MOE_DEVCACHE_DMA=0\nelse\n  export LLAMA_MOE_DEVCACHE_DMA=1 CUDA_DEVICE_MAX_CONNECTIONS=32\nfi\nTHREADS=$(nproc 2>/dev/null || echo 16); [ "$THREADS" -gt 24 ] && THREADS=24\nARGS=(-m /content/model/DeepSeek-V4.1-Flash-1017-GSQ-RCO-$RUNG-00001-of-00006.gguf -a ds41,ds41-max --host 127.0.0.1 --port $PORT\n  -ngl 99 -fit off -ot "$CPU_RE" -fa on -c $CTX -np 2 -t "$THREADS" -b $UBATCH -ub $UBATCH -lv 4 -lm mmap --jinja --reasoning on --reasoning-effort 50\n  --metrics --api-key-file /content/secrets/key --cache-ram $CACHE_RAM --moe-cache-mib $MOE_CACHE_MIB)\n[ "$DRAFT" = 1 ] && ARGS+=(-md /mnt/ram/m/dspark.gguf --spec-type draft-dspark --spec-draft-n-max 2 -ngld 99 --spec-draft-n-ubatch 512\n  --no-spec-draft-backend-sampling)  # backend draft sampling garbles the second sequence of a shared draft batch\n[ "$MMPROJ" = 1 ] && ARGS+=(--mmproj $MMP)\necho "$BIN ${ARGS[*]}" > /content/res/serve_cmd.txt\ncd "$(dirname "$BIN")"\nrm -f /content/res/pin_result /content/res/pin_request\nLD_PRELOAD=/content/vm/libpin.so LD_LIBRARY_PATH="$(dirname "$BIN"):/usr/lib64-nvidia" setsid nohup "$BIN" "${ARGS[@]}" > /content/res/server.log 2>&1 < /dev/null &\necho $! > /content/res/server.pid\n'

for name, text in [("boot.sh", BOOT_SH), ("build.sh", BUILD_SH), ("serve.sh", SERVE_SH)]:
    p = pathlib.Path("/content/vm") / name
    p.write_text(text.replace("\r\n", "\n"), encoding="utf-8")
    os.chmod(p, 0o755)

pathlib.Path("/content/secrets").mkdir(parents=True, exist_ok=True)
key = API_KEY.strip() or ("sk-ds41-" + secrets.token_hex(24))
pathlib.Path("/content/secrets/key").write_text(key, encoding="utf-8")
if HF_TOKEN.strip():
    pathlib.Path("/content/secrets/hf_hdr").write_text(f"Authorization: Bearer {HF_TOKEN.strip()}", encoding="utf-8")

images = False if FAST_MODE else bool(IMAGES)
ctx_req = min(int(CONTEXT_TOKENS), 131072) if FAST_MODE else int(CONTEXT_TOKENS)
fast_layout = (not images) and (ctx_req <= 131072)
env = os.environ.copy()
env.update({
    "CTX": str(2 * ctx_req),
    "UBATCH": "3072" if fast_layout else "2048",
    "MMPROJ": "1" if images else "0",
    "MOE_CACHE_MIB": "74231" if fast_layout else "70430",
    "DYN": "166" if fast_layout else "110",
    "DMA": "1" if fast_layout else "0",
    "DRAFT": "1",
    "UNCENSOR": "1" if UNCENSOR else "0",
    "WORKERS": "64",
})

pathlib.Path("/content/res").mkdir(parents=True, exist_ok=True)
tl_path = pathlib.Path("/content/res/timeline.txt")
tl_path.write_text("", encoding="utf-8")
print("\n=== Booting DeepSeek V4.1 Flash (about 4-6 minutes) ===", flush=True)
boot_proc = subprocess.Popen(["bash", "/content/vm/boot.sh"], env=env)
seen = 0
while boot_proc.poll() is None:
    lines = tl_path.read_text(encoding="utf-8", errors="replace").splitlines()
    for l in lines[seen:]:
        print("  " + l, flush=True)
    seen = len(lines)
    time.sleep(2)
lines = tl_path.read_text(encoding="utf-8", errors="replace").splitlines()
for l in lines[seen:]:
    print("  " + l, flush=True)
if boot_proc.returncode != 0:
    print("\n--- logs tail ---")
    sh("tail -n 40 /content/res/server.log /content/res/build.txt /content/res/build_make.log 2>/dev/null", check=False)
    raise RuntimeError(f"boot.sh failed with exit code {boot_proc.returncode}")

if not shutil.which("cloudflared"):
    sh("curl -sL https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -o /usr/local/bin/cloudflared && chmod +x /usr/local/bin/cloudflared")
cf_log = pathlib.Path("/content/res/cloudflared.log")
sh("pkill -f 'cloudflared tunnel' || true", check=False)
subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:18080", "--no-autoupdate"],
    stdout=open(cf_log, "w"),
    stderr=subprocess.STDOUT,
)
public_url = None
for _ in range(30):
    txt = cf_log.read_text(encoding="utf-8", errors="replace")
    m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com", txt)
    if m:
        public_url = m.group(0)
        break
    time.sleep(1)

print("\n" + "=" * 60)
print("READY - DeepSeek V4.1 Flash (Abliterated)")
print(f"  Base URL : {public_url}/v1" if public_url else "  Local URL: http://127.0.0.1:18080/v1 (check /content/res/cloudflared.log)")
print(f"  API Key  : {key}")
print("  Models   : ds41 (effort 50), ds41-max (effort max)")
print("=" * 60)


In [ ]:
#@title 2. Quick Test Request
import json, pathlib, urllib.request

key = pathlib.Path("/content/secrets/key").read_text().strip()
body = json.dumps({
    "model": "ds41",
    "messages": [{"role": "user", "content": "Say hello and state your model name in one sentence."}],
    "max_tokens": 128,
}).encode()
req = urllib.request.Request(
    "http://127.0.0.1:18080/v1/chat/completions",
    data=body,
    headers={"Authorization": f"Bearer {key}", "Content-Type": "application/json"},
)
with urllib.request.urlopen(req, timeout=120) as r:
    data = json.loads(r.read().decode())
print(data["choices"][0]["message"].get("content", ""))
if "timings" in data:
    print("\nSpeed:", round(data["timings"].get("predicted_per_second", 0), 1), "tok/s")
